# 15.9 The cold-start problem

Cold start is not one problem: new users, new items, and new contexts each remove a different source of evidence.

This gap-topic notebook treats missing behavior as uncertainty rather than dislike. It combines Beta-binomial shrinkage, content cosine fallback, and behavior blending so long-tail and new inventory can earn exposure.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(1507)


def softmax(scores, temperature=1.0):
    values = np.asarray(scores, dtype=float) / float(temperature)
    values = values - np.max(values)
    weights = np.exp(values)
    return weights / np.sum(weights)


def normalize_rows(matrix):
    values = np.asarray(matrix, dtype=float)
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return values / norms


def top_k_indices(scores, k):
    values = np.asarray(scores, dtype=float)
    order = np.argsort(-values)
    return order[:k]


def recall_at_k(scores, positives, k=3):
    values = np.asarray(scores, dtype=float)
    hits = []
    for row, pos in zip(values, positives):
        top = set(top_k_indices(row, k))
        wanted = set(np.atleast_1d(pos).astype(int))
        hits.append(len(top & wanted) / max(1, len(wanted)))
    return float(np.mean(hits))


def dcg_at_k(relevance, k=3):
    rel = np.asarray(relevance, dtype=float)[:k]
    gains = np.power(2.0, rel) - 1.0
    discounts = np.log2(np.arange(2, len(rel) + 2))
    return float(np.sum(gains / discounts))


def ndcg_at_k(scores, relevance, k=3):
    values = np.asarray(scores, dtype=float)
    rel = np.asarray(relevance, dtype=float)
    order = np.argsort(-values)[:k]
    ideal = np.argsort(-rel)[:k]
    ideal_dcg = dcg_at_k(rel[ideal], k)
    if ideal_dcg == 0.0:
        return 0.0
    return dcg_at_k(rel[order], k) / ideal_dcg


def mrr_from_scores(scores, target):
    order = np.argsort(-np.asarray(scores, dtype=float))
    rank = int(np.where(order == int(target))[0][0]) + 1
    return 1.0 / rank


def make_f14_ladder(seed=1514):
    rng = np.random.default_rng(seed)
    rungs = []

    item_vectors = np.array([
        [1.0, 0.1, 0.0],
        [0.8, 0.4, 0.0],
        [0.1, 1.0, 0.2],
        [-0.2, 0.3, 0.9],
    ])
    user_vectors = np.array([
        [1.0, 0.2, 0.0],
        [0.2, 1.0, 0.2],
        [-0.1, 0.4, 1.0],
    ])
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D1 tiny slate",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.array([0]), np.array([2]), np.array([3])],
        "sessions": [[0, 1, 2], [1, 2, 0], [2, 3, 1]],
        "targets": [2, 0, 1],
        "clicks": np.array([5, 8, 1]),
        "impressions": np.array([100, 100, 20]),
        "eligible": np.array([True, True, True, True]),
        "exposure": np.array([1.0, 0.8, 0.5, 0.3]),
    })

    angles = np.linspace(0.0, 2.0 * np.pi, 8, endpoint=False)
    item_vectors = np.column_stack([np.cos(angles), np.sin(angles), 0.35 * np.cos(2.0 * angles)])
    user_angles = np.array([0.1, 1.6, 3.0, 4.7, 5.6])
    user_vectors = np.column_stack([np.cos(user_angles), np.sin(user_angles), 0.25 * np.sin(user_angles)])
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D2 synthetic preferences",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:2] for row in relevance],
        "sessions": [[0, 1, 2, 3], [2, 3, 4], [4, 5, 6], [6, 7, 0], [7, 0, 1]],
        "targets": [3, 4, 6, 0, 1],
        "clicks": np.array([12, 20, 18, 6, 4, 3, 9, 7]),
        "impressions": np.array([200, 260, 220, 150, 80, 60, 120, 140]),
        "eligible": np.ones(8, dtype=bool),
        "exposure": np.linspace(1.0, 0.35, 8),
    })

    item_vectors = rng.normal(size=(12, 4))
    item_vectors = normalize_rows(item_vectors)
    user_vectors = rng.normal(size=(7, 4))
    user_vectors = normalize_rows(user_vectors)
    relevance = user_vectors @ item_vectors.T
    relevance = relevance + rng.normal(scale=0.04, size=relevance.shape)
    exposure = rng.uniform(0.15, 1.0, size=12)
    observed = rng.uniform(size=relevance.shape) < exposure[None, :]
    sparse_relevance = np.where(observed, relevance, -0.2)
    rungs.append({
        "name": "D3 sparse noisy logs",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": sparse_relevance,
        "true_relevance": relevance,
        "positives": [np.argsort(-row)[:2] for row in relevance],
        "sessions": [[0, 2, 5, 7], [1, 2, 4], [3, 8, 9], [10, 2, 0], [6, 6, 11], [4, 5, 6], [9, 1, 3]],
        "targets": [7, 4, 9, 0, 11, 6, 3],
        "clicks": np.array([30, 16, 9, 4, 3, 2, 8, 7, 5, 4, 2, 1]),
        "impressions": np.array([600, 280, 190, 90, 45, 30, 120, 110, 70, 65, 22, 12]),
        "eligible": rng.uniform(size=12) > 0.10,
        "exposure": exposure,
    })

    genres = np.array([
        [1, 0, 0, 0, 1],
        [1, 1, 0, 0, 0],
        [0, 1, 1, 0, 0],
        [0, 0, 1, 1, 0],
        [0, 0, 0, 1, 1],
        [1, 0, 0, 1, 0],
        [0, 1, 0, 0, 1],
        [0, 0, 1, 0, 1],
        [1, 0, 1, 0, 0],
        [0, 1, 0, 1, 0],
    ], dtype=float)
    item_vectors = normalize_rows(genres + 0.05 * rng.normal(size=genres.shape))
    user_vectors = normalize_rows(np.array([
        [2, 0, 0, 0, 1],
        [1, 2, 0, 0, 0],
        [0, 1, 2, 0, 0],
        [0, 0, 1, 2, 0],
        [0, 0, 0, 1, 2],
        [1, 0, 1, 0, 1],
    ], dtype=float))
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D4 MovieLens-like inline",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:3] for row in relevance],
        "sessions": [[0, 5, 8], [1, 2, 6], [2, 3, 8], [3, 4, 9], [4, 7, 0], [7, 8, 6]],
        "targets": [8, 6, 3, 9, 4, 6],
        "clicks": np.array([50, 34, 25, 18, 17, 14, 9, 7, 6, 5]),
        "impressions": np.array([1000, 700, 520, 430, 320, 250, 150, 110, 90, 80]),
        "eligible": np.ones(10, dtype=bool),
        "exposure": np.array([1.0, 0.9, 0.75, 0.65, 0.55, 0.45, 0.35, 0.30, 0.25, 0.20]),
    })

    head = normalize_rows(rng.normal(loc=0.6, scale=0.25, size=(5, 5)))
    torso = normalize_rows(rng.normal(loc=0.0, scale=0.7, size=(8, 5)))
    cold = normalize_rows(np.array([
        [1, 1, 0, 0, 1],
        [0, 1, 1, 1, 0],
        [1, 0, 1, 0, 1],
        [0, 0, 1, 1, 1],
    ], dtype=float))
    item_vectors = np.vstack([head, torso, cold])
    user_vectors = normalize_rows(rng.normal(size=(8, 5)))
    user_vectors[0] = normalize_rows(np.array([[1, 1, 0, 0, 1]], dtype=float))[0]
    relevance = user_vectors @ item_vectors.T
    popularity = np.array([2000, 1500, 1200, 900, 700, 250, 180, 130, 100, 75, 60, 45, 35, 10, 8, 5, 3])
    clicks = np.maximum(1, np.round(popularity * np.clip(0.03 + 0.04 * np.maximum(item_vectors[:, 0], 0.0), 0.01, 0.20))).astype(int)
    exposure = np.clip(popularity / popularity.max(), 0.02, 1.0)
    rungs.append({
        "name": "D5 long-tail cold-start",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:3] for row in relevance],
        "sessions": [[0, 1, 14], [1, 6, 15], [2, 8, 16], [3, 9, 13], [4, 10, 14], [5, 11, 15], [6, 12, 16], [7, 13, 14]],
        "targets": [14, 15, 16, 13, 14, 15, 16, 14],
        "clicks": clicks,
        "impressions": popularity.astype(int),
        "eligible": np.array([True] * 13 + [True, True, False, True]),
        "exposure": exposure,
        "cold_items": np.array([13, 14, 15, 16]),
    })

    return rungs


## The concept, built once: shrink, fall back, blend

Raw popularity is $\hat p=c/n$, but a Beta-binomial prior gives

$$\mathbb{E}[p\mid c,n]=\frac{c+\alpha}{n+\alpha+\beta}.$$

The reusable scorer combines shrunk CTR, content cosine, and a user-history blend weight $w=n/(n+5)$.

In [ ]:

def cold_start_score(clicks, impressions, item_vectors, user_profile, behavior_scores=None, alpha=10.0, beta=190.0, events=0):
    clicks = np.asarray(clicks, dtype=float)
    impressions = np.asarray(impressions, dtype=float)
    raw_ctr = clicks / np.maximum(impressions, 1.0)
    shrunk_ctr = (clicks + alpha) / (impressions + alpha + beta)
    items = normalize_rows(item_vectors)
    profile = np.asarray(user_profile, dtype=float)
    profile = profile / max(np.linalg.norm(profile), 1e-12)
    content = items @ profile
    content = (content + 1.0) / 2.0
    if behavior_scores is None:
        behavior_scores = shrunk_ctr
    behavior_scores = np.asarray(behavior_scores, dtype=float)
    weight = float(events / (events + 5.0))
    blended = (1.0 - weight) * content + weight * behavior_scores
    final = 0.45 * shrunk_ctr + 0.55 * blended
    return {
        "raw_ctr": raw_ctr,
        "shrunk_ctr": shrunk_ctr,
        "content": content,
        "weight": weight,
        "score": final,
    }


## Verify the lesson numbers once

Clicks $(50,20,5)$ over impressions $(1000,200,20)$ produce raw CTRs $0.050$, $0.100$, $0.250$. With $\alpha=10$, $\beta=190$, shrinkage gives $0.050$, $0.075$, $0.068$. A new item $(1,1)$ and profile $(1,0.5)$ have cosine $0.949$. With $w=n/(n+5)$ at $n=20$, $(1-w)0.06+w0.18=0.156$, which is above $0.150$.

In [ ]:

clicks = np.array([50, 20, 5])
impressions = np.array([1000, 200, 20])
raw = clicks / impressions
shrunk = (clicks + 10.0) / (impressions + 200.0)
assert [round(float(x), 3) for x in raw] == [0.050, 0.100, 0.250]
assert [round(float(x), 3) for x in shrunk] == [0.050, 0.075, 0.068]

new_item = np.array([1.0, 1.0])
profile = np.array([1.0, 0.5])
cosine = float(new_item @ profile / (np.linalg.norm(new_item) * np.linalg.norm(profile)))
assert round(cosine, 3) == 0.949

n_events = 20
weight = n_events / (n_events + 5.0)
blend = (1.0 - weight) * 0.06 + weight * 0.18
assert round(blend, 3) == 0.156
assert blend > 0.150
print("raw", np.round(raw, 3), "shrunk", np.round(shrunk, 3), "cosine", round(cosine, 3), "blend", round(blend, 3))


## The dataset ladder: D1 to D5

The same F14 recommender ladder is built inline: tiny slate, synthetic preferences, sparse noisy logs, a MovieLens-like genre matrix, and a long-tail cold-start catalog. The single metric here is **NDCG@3**.

In [ ]:

rungs = make_f14_ladder()
for rung in rungs:
    user_shape = rung["user_vectors"].shape
    item_shape = rung["item_vectors"].shape
    rel_shape = rung["relevance"].shape
    sparsity = float(np.mean(rung.get("exposure", np.ones(item_shape[0])) < 0.5))
    print(rung["name"], "users", user_shape, "items", item_shape, "relevance", rel_shape, "low exposure fraction", round(sparsity, 3))
    print("sample relevance row", np.round(rung["relevance"][0, : min(5, item_shape[0])], 3))


## Run the same method across D1-D5

Each rung ranks items with shrinkage, content fallback, and a behavior blend. NDCG@3 uses the rung's true relevance, so cold-start improvements are visible when content helps relevant tail items appear early.

In [ ]:

results = []
for rung in rungs:
    profile = rung["user_vectors"][0]
    output = cold_start_score(rung["clicks"], rung["impressions"], rung["item_vectors"], profile, behavior_scores=rung["relevance"][0], events=20)
    metric = ndcg_at_k(output["score"], rung["relevance"][0], k=3)
    results.append({"name": rung["name"], "metric": metric, "output": output})

for row in results:
    print(f"{row['name']:<28} NDCG@3={row['metric']:.3f}")


## Results visualization

Panels compare shrunk prior, content fallback, and final score for each rung. The curve shows NDCG@3 from the tiny slate through long-tail cold start.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()
for ax, rung, row in zip(flat_axes[:5], rungs, results):
    output = row["output"]
    order = np.argsort(-output["score"])[:5]
    ax.plot(output["shrunk_ctr"][order], marker="o", label="prior")
    ax.plot(output["content"][order], marker="s", label="content")
    ax.plot(output["score"][order], marker="^", label="final")
    ax.set_title(rung["name"])
    ax.set_xlabel("ranked candidate")
    ax.set_ylabel("score")
flat_axes[0].legend(fontsize=8)
flat_axes[5].plot(range(1, 6), [row["metric"] for row in results], marker="o")
flat_axes[5].set_xticks(range(1, 6))
flat_axes[5].set_xticklabels(["D1", "D2", "D3", "D4", "D5"])
flat_axes[5].set_ylim(0.0, 1.05)
flat_axes[5].set_title("NDCG@3 curve")
fig.tight_layout()


## Pitfall on D5: no data is not bad data

Ranking raw CTR with tiny denominators over-ranks noisy items, while treating new items as zero creates a self-fulfilling failure. The fix combines shrinkage, content fallback, and allocated exposure.

In [ ]:

d5 = rungs[-1]
profile = d5["user_vectors"][0]
raw_output = cold_start_score(d5["clicks"], d5["impressions"], d5["item_vectors"], profile, behavior_scores=np.zeros_like(d5["clicks"], dtype=float), events=0)
wrong_scores = raw_output["raw_ctr"]
fixed_output = cold_start_score(d5["clicks"], d5["impressions"], d5["item_vectors"], profile, behavior_scores=d5["relevance"][0], events=20)
wrong_metric = ndcg_at_k(wrong_scores, d5["relevance"][0], k=3)
fixed_metric = ndcg_at_k(fixed_output["score"], d5["relevance"][0], k=3)
print("raw CTR NDCG@3", round(wrong_metric, 3))
print("shrinkage plus content NDCG@3", round(fixed_metric, 3))
print("cold item scores", np.round(fixed_output["score"][d5["cold_items"]], 3))


## Evaluate it + practice

- Report **NDCG@3** against a no-skill popularity or random baseline on every rung.
- Sanity check that the D1 worked numbers match the lesson asserts before trusting larger rungs.
- Ablation: remove the content fallback for items with fewer than 20 impressions; the metric should drop on D3-D5.
- Failure signals: head-item collapse, cold items never appearing, or a metric curve that improves only because labels are biased.
- Keep splits chronological or exposure-aware when the lesson's pitfall involves time or logging.

Practice: change the cutoff from 3 to 5 and compare the metric curve.

Practice: change the Beta prior strength and watch which D5 tail items move.